# Skeleton Animatronic: Low-Latency Dialogue & Audio Subsystem Demo

This interactive notebook demonstrates the **low-latency context-specific dialogue engine** and **audio subsystem** for the animatronic skeleton.

### Key Capabilities Demonstrated:
1. **Multimodal Sensory Context**: Ingests vision tracking (head angles, subject distance, detected objects) alongside audio transcripts.
2. **Streaming Clause Chunking**: Streams token fragments directly into natural speech boundaries so downstream TTS & mouth sync begin playing on the very first clause without waiting for full completion.
3. **Telemetry & Latency Budget**: Measures **Time-To-First-Token (TTFT)** and **Time-To-First-Chunk (TTFC)** to meet animatronic responsiveness ($<500\text{ ms}$).
4. **Barge-In Interruption Handling**: Instantly cancels in-flight LLM generation and motor queues when the user speaks again.
5. **Phonetic Sanitizer**: Automatically purges stage directions (`*cackles*`, `[whispers]`), asterisks, and emojis that would corrupt TTS pronunciation and lip-sync.
6. **Interactive Chat**: GUI widget and direct Python cell chat.
7. **Deep Male Skeleton Voice (TTS)**: Synthesizes speech using OpenAI `tts-1` with voice `onyx` at $0.90\times$ speed for an imposing, sinister cadence.
8. **Mechanical Mouth Sync**: Extracts 30Hz jaw-opening servo angles with deadband noise gate to animate physical skeleton skull servos without motor jitter.

In [1]:
import asyncio
import os
import sys
import time
from pathlib import Path

# Ensure the skeleton package is on the python path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from skeleton.dialogue.config import DialogueConfig
from skeleton.dialogue.models import DialogueState, SpeechChunk, VisionContext
from skeleton.dialogue.llm_client import LowLatencyLLMClient, MockLLMClient
from skeleton.dialogue.prompt_builder import PromptBuilder
from skeleton.dialogue.sanitizer import TextSanitizer
from skeleton.dialogue.sentence_chunker import SentenceChunker
from skeleton.dialogue.orchestrator import DialogueOrchestrator

print("Skeleton Dialogue Engine successfully imported!")

Skeleton Dialogue Engine successfully imported!


## 1. Engine Initialization (Google Gemini API)

This demo is configured to stream live dialogue from the **Google Gemini API** (`gemini-3.6-flash`).
- Keys are loaded automatically from `.env` (`GEMINI_API_KEY`).
- The engine streams via Google's OpenAI-compatible endpoint with thinking latency disabled (`reasoning_effort="none"`) for rapid spoken banter.
- If no API key is available, it gracefully falls back to the local deterministic `MockLLMClient`.

In [2]:
from dotenv import load_dotenv
load_dotenv(project_root / ".env")

config = DialogueConfig()
has_api_key = config.api_key is not None and config.api_key.get_secret_value() not in ("", "mock-or-env-key")

if has_api_key:
    print(f"[Mode: Live API]")
    print(f"Model:    {config.model}")
    print(f"Endpoint: {config.base_url}")
    llm_client = LowLatencyLLMClient(config)
else:
    print("[Mode: Offline Mock] Using MockLLMClient with simulated 35ms TTFT and 15ms inter-token stream.")
    llm_client = MockLLMClient(
        simulated_response="Well, look who decided to wander into my lair. Nice shoes, do they come with a personality?",
        ttft_delay_s=0.035,
        inter_token_delay_s=0.015,
        config=config,
    )

orchestrator = DialogueOrchestrator(config=config, llm_client=llm_client)
print("Dialogue Orchestrator ready!")

[Mode: Live API]
Model:    gemini-3.6-flash
Endpoint: https://generativelanguage.googleapis.com/v1beta/openai/
Dialogue Orchestrator ready!


## 2. Interactive Multimodal Dialogue & Streaming Telemetry

Let's simulate an interaction where the skeleton sees a visitor holding a coffee cup looking to the right, and the audio receiver transcribes the user's greeting.

In [3]:
# Define simulated vision tracking state
vision_state = VisionContext(
    subject_detected=True,
    pan_angle_deg=25.0,  # 25 degrees to the right
    tilt_angle_deg=-5.0,
    distance_m=1.8,
    detected_objects=["coffee mug", "headphones"],
    subject_facing_skeleton=False,  # looking away
)

user_speech = "Hey there, can you see what I'm holding?"

print("--- Vision Context Summary ---")
print(vision_state.to_prompt_string())
print("\n--- User Utterance ---")
print(f"User: \"{user_speech}\"")

--- Vision Context Summary ---
Subject is 25.0° right, looking away. approx 1.8m away. detected items: coffee mug, headphones.

--- User Utterance ---
User: "Hey there, can you see what I'm holding?"


In [4]:
async def run_dialogue_stream(user_text, vision):
    print("\n--- Live Chunk Stream & Latency Telemetry ---")
    chunk_count = 0
    first_chunk_ms = None
    
    async for chunk in orchestrator.process_utterance(user_text, vision):
        chunk_count += 1
        if first_chunk_ms is None:
            first_chunk_ms = chunk.elapsed_ms
        
        tag = "[FINAL]" if chunk.is_final else f"[CHUNK {chunk.sequence_index}]"
        print(f"{tag} (+{chunk.elapsed_ms:6.1f}ms | {chunk.word_count:2d} words): \"{chunk.text}\"")
    
    print("--------------------------------------------")
    if first_chunk_ms is not None:
        print(f"Time to First Chunk (TTS ready): {first_chunk_ms:.1f} ms")
    print(f"Total Chunks Emitted: {chunk_count}")

# Execute dialogue in notebook event loop
await run_dialogue_stream(user_speech, vision_state)


--- Live Chunk Stream & Latency Telemetry ---
[CHUNK 0] (+1049.3ms |  4 words): "Ah, a coffee mug."
[CHUNK 1] (+1049.5ms | 11 words): "Fueling a body I do not have, while ignoring me entirely."
[CHUNK 2] (+1122.9ms |  2 words): "How charming."
--------------------------------------------
Time to First Chunk (TTS ready): 1049.3 ms
Total Chunks Emitted: 3


## 3. Barge-In Interruption Testbed

When a user interrupts while the skeleton is speaking, the orchestrator cancels the active task and immediately begins processing the new utterance without motor thrashing or double-speaking.

In [5]:
# Configure a slow streaming mock to simulate the skeleton speaking a longer remark
slow_mock = MockLLMClient(
    simulated_response="I was in the middle of a very witty soliloquy about calcium when you rudely barged in.",
    ttft_delay_s=0.03,
    inter_token_delay_s=0.08,
)
barge_in_orchestrator = DialogueOrchestrator(config=config, llm_client=slow_mock)

print("[1] User speaks: 'Tell me a story...'")
stream = barge_in_orchestrator.process_utterance("Tell me a story")

# Receive the first chunk
first_chunk = await anext(stream)
print(f"Skeleton started speaking: \"{first_chunk.text}\"")

print("\n[2] User abruptly interrupts: 'Stop, never mind!'")
# Interruption triggered by new utterance call
slow_mock.simulated_response = "Fine, I have better things to haunt anyway."
slow_mock.inter_token_delay_s = 0.01

async for chunk in barge_in_orchestrator.process_utterance("Stop, never mind!"):
    print(f"Interrupted Recovery Chunk: \"{chunk.text}\"")

print("\nBarge-In successfully handled with clean cancellation!")

[1] User speaks: 'Tell me a story...'
Skeleton started speaking: "I was in the middle of a very witty soliloquy about calcium when you rudely barged in."

[2] User abruptly interrupts: 'Stop, never mind!'
Interrupted Recovery Chunk: "Fine,"
Interrupted Recovery Chunk: "I have better things to haunt anyway."

Barge-In successfully handled with clean cancellation!


## 4. Phonetic & Lip-Sync Sanitizer Inspection

Verifying that stage directions, asterisks, and emojis are stripped so the physical mouth servo doesn't desynchronize or speak 'asterisk'.

In [6]:
raw_llm_outputs = [
    "*cackles dryly* You really think you can defeat me? 💀",
    "[whispering softly] Don't look behind you, fleshy.",
    "I have **two hundred and six** bones, and `zero` patience.",
    "Oh please... (clears non-existent throat) you call that dancing? 😂"
]

print("RAW MODEL OUTPUT -> SANITIZED FOR TTS & LIP-SYNC")
print("=" * 60)
for raw in raw_llm_outputs:
    clean = TextSanitizer.sanitize(raw)
    print(f"Raw:   {raw}")
    print(f"Clean: {clean}")
    print("-" * 60)

RAW MODEL OUTPUT -> SANITIZED FOR TTS & LIP-SYNC
Raw:   *cackles dryly* You really think you can defeat me? 💀
Clean: You really think you can defeat me?
------------------------------------------------------------
Raw:   [whispering softly] Don't look behind you, fleshy.
Clean: Don't look behind you, fleshy.
------------------------------------------------------------
Raw:   I have **two hundred and six** bones, and `zero` patience.
Clean: I have two hundred and six bones, and zero patience.
------------------------------------------------------------
Raw:   Oh please... (clears non-existent throat) you call that dancing? 😂
Clean: Oh please... you call that dancing?
------------------------------------------------------------


## 5. Live Interactive Chat

You have two ways to talk to the skeleton below:
- **Option A (Widget UI)**: Type in the box and click **"Talk to Skeleton"**.
- **Option B (Direct Python Cell)**: If your browser blocks Jupyter widget buttons, simply run the Python cell in 5.2!

In [7]:
# 5.1 Interactive Chat Widget
import ipywidgets as widgets
from IPython.display import display

input_box = widgets.Text(
    value="Who let you out of the closet?",
    placeholder="Say something to the skeleton...",
    description="You:",
    layout=widgets.Layout(width="70%"),
    continuous_update=True,
)

send_btn = widgets.Button(
    description="Talk to Skeleton",
    button_style="danger",
    icon="comment",
)

chat_output = widgets.Output()

async def _stream_response(user_text):
    try:
        chat_output.clear_output(wait=True)
        chat_output.append_stdout(f"You: \"{user_text}\"\n")
        chat_output.append_stdout("Skeleton: ")
        async for chunk in orchestrator.process_utterance(user_text, vision_context=vision_state):
            chat_output.append_stdout(f"{chunk.text} ")
        chat_output.append_stdout("\n\n")
    except Exception as exc:
        chat_output.append_stderr(f"\n[Error] {exc}\n")

def _on_submit(b=None):
    text = input_box.value.strip()
    if not text:
        return
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = asyncio.get_event_loop()
    loop.create_task(_stream_response(text))

send_btn.on_click(_on_submit)

# Display the widget
display(widgets.HBox([input_box, send_btn]), chat_output)

# Run once on cell execution to demo immediately
await _stream_response(input_box.value)

Output()

### 5.2 Direct Python Chat (Zero-Widget Fallback)

If your browser's Jupyter environment blocks or disconnects widget event listeners, you can talk to the skeleton directly from Python:
Simply change `my_message` below and press **Shift + Enter** to run!

In [8]:
# Type your custom message here and press Shift + Enter:
my_message = "with my music, I don't have to listen to nonsense like you"

async def talk(text: str):
    print(f"You: \"{text}\"\n")
    print("Skeleton: ", end="", flush=True)
    async for chunk in orchestrator.process_utterance(text, vision_context=vision_state):
        print(f"{chunk.text} ", end="", flush=True)
    print("\n")

await talk(my_message)

You: "with my music, I don't have to listen to nonsense like you"

Skeleton: Oh, keep those headphones on then. Wouldn't want my brilliant bone-marrow-deep wisdom disrupting your terrible taste in music. 



## 6. Deep Male Skeleton Voice (TTS), Whisper STT & Mechanical Mouth Sync

This section implements the physical voice and jaw actuation pipeline:
- **Voice Profile**: OpenAI `tts-1` with voice `onyx` at $0.90\times$ speed for an imposing, sinister gravelly cadence.
- **Mouth Sync Generation**: Analyzes uncompressed 16-bit WAV PCM audio to extract a continuous 30Hz jaw servo trajectory ($0.0^\circ$ to $35.0^\circ$) with a deadband noise gate to protect mechanical gears.

In [9]:
# 6.1 Synthesize Deep Male Voice & Play in Notebook
from IPython.display import Audio, display
from skeleton.audio.tts import OpenAITTSClient, MockTTSClient
from skeleton.audio.mouth_sync import MouthSyncProcessor

tts_client = OpenAITTSClient() if os.environ.get("OPENAI_API_KEY") else MockTTSClient()
mouth_sync = MouthSyncProcessor()

sample_speech = "Greetings, mortal. I may be completely hollow, but I have plenty of backbone."
print(f"Synthesizing in deep 'onyx' voice: \"{sample_speech}\"")

wav_bytes = await tts_client.synthesize(sample_speech)
jaw_frames = mouth_sync.extract_jaw_trajectory(wav_bytes)

print(f"Audio Generated: {len(wav_bytes):,} bytes (WAV PCM)")
print(f"Computed {len(jaw_frames)} jaw motion frames at 30 fps.")
peak_angle = max(f.jaw_angle_deg for f in jaw_frames) if jaw_frames else 0.0
print(f"Peak Jaw Opening: {peak_angle:.1f}°\n")

# Listen to the deep skeleton voice in the browser:
display(Audio(wav_bytes, autoplay=False))

Synthesizing in deep 'onyx' voice: "Greetings, mortal. I may be completely hollow, but I have plenty of backbone."
Audio Generated: 249,378 bytes (WAV PCM)
Computed 156 jaw motion frames at 30 fps.
Peak Jaw Opening: 13.0°



In [10]:
# 6.2 Visualizing Jaw Movement Timeline (Servo Trajectory)
print(f"{'Timestamp':>12} | {'Jaw Angle':>10} | {'Jaw Position Visualizer':<25}")
print("-" * 54)
for frame in jaw_frames[:16]:
    bar = "█" * int(frame.normalized_open * 22)
    print(f"{frame.timestamp_ms:10.1f}ms | {frame.jaw_angle_deg:9.1f}° | {bar:<25}")

   Timestamp |  Jaw Angle | Jaw Position Visualizer  
------------------------------------------------------
       0.0ms |       0.0° |                          
      33.3ms |       0.2° |                          
      66.7ms |       5.5° | ███                      
     100.0ms |       9.2° | █████                    
     133.3ms |      10.2° | ██████                   
     166.7ms |       6.4° | ████                     
     200.0ms |       5.7° | ███                      
     233.3ms |       4.8° | ███                      
     266.7ms |       3.7° | ██                       
     300.0ms |       2.7° | █                        
     333.3ms |       1.8° | █                        
     366.7ms |       0.7° |                          
     400.0ms |       0.3° |                          
     433.3ms |       0.1° |                          
     466.7ms |       0.1° |                          
     500.0ms |       0.4° |                          


In [11]:
# 6.3 Complete End-to-End Turn: Dialogue (Gemini) -> Deep TTS (Onyx) -> Mouth Sync
from skeleton.audio.pipeline import SkeletonAudioPipeline
from skeleton.audio.stt import WhisperSTTClient, MockSTTClient

stt_client = WhisperSTTClient() if os.environ.get("OPENAI_API_KEY") else MockSTTClient()
audio_pipeline = SkeletonAudioPipeline(
    stt_client=stt_client,
    tts_client=tts_client,
    orchestrator=orchestrator,
    mouth_sync=mouth_sync,
)

user_query = "Do you have any plans for Halloween?"
print(f"User Input: \"{user_query}\"")

response = await audio_pipeline.process_text_turn(user_query, vision_context=vision_state)
print(f"\nSkeleton Answered: \"{response.skeleton_response}\"")
print(f"Turnaround Latency: {response.total_latency_ms:.1f} ms")
print(f"Synchronized Mouth Frames: {len(response.mouth_frames)}\n")

# Listen to the response audio:
display(Audio(response.audio_bytes, autoplay=False))

User Input: "Do you have any plans for Halloween?"

Skeleton Answered: "Just planning to sit here and judge your outfit, though sipping from that mug over there seems much less dead-boring."
Turnaround Latency: 2503.7 ms
Synchronized Mouth Frames: 239



In [12]:
vision_state = VisionContext(
    subject_detected=True,
    pan_angle_deg=25.0,  # 25 degrees to the right
    tilt_angle_deg=-5.0,
    distance_m=1.8,
    detected_objects=["green sweatshirt", "orange bucket"],
    subject_facing_skeleton=True,  # looking away
)

In [13]:
user_query = "what are you looking at?"
print(f"User Input: \"{user_query}\"")

response = await audio_pipeline.process_text_turn(user_query, vision_context=vision_state)
print(f"\nSkeleton Answered: \"{response.skeleton_response}\"")
print(f"Turnaround Latency: {response.total_latency_ms:.1f} ms")
print(f"Synchronized Mouth Frames: {len(response.mouth_frames)}\n")

# Listen to the response audio:
display(Audio(response.audio_bytes, autoplay=False))

User Input: "what are you looking at?"

Skeleton Answered: "I am looking at that glowing orange bucket you are holding. Planning to collect a skeleton crew or just hoard all the candy?"
Turnaround Latency: 2717.8 ms
Synchronized Mouth Frames: 239



### 6.4 Live Microphone Audio Recording & Speech Processing

Capture voice directly from your physical microphone (webcam microphone, USB mic, headset, or system default):
1. User speech is recorded at **16kHz 16-bit mono PCM WAV** (Whisper native acoustic rate).
2. **OpenAI Whisper STT** transcribes the recorded speech into text.
3. **Google Gemini API** (`gemini-3.6-flash`) evaluates user prompt with real-time vision context and generates a snarky skeleton response.
4. **OpenAI TTS** (`onyx` at $0.90\times$ speed) synthesizes the deep gravelly voice in WAV format.
5. **Mouth Sync Engine** generates a 30Hz servo jaw trajectory ($0^\circ - 35^\circ$) with deadband protection.


In [14]:
# 6.4.1 Query Physical Audio Input Devices
from skeleton.audio.recorder import AudioRecorder, MockAudioRecorder, AudioDeviceError

recorder = AudioRecorder()
try:
    input_devices = recorder.list_input_devices()
    print(f"Discovered {len(input_devices)} Physical Audio Input Device(s):\n")
    for dev in input_devices:
        print(f"  [Index {dev.index}] {dev.name} ({dev.max_input_channels} ch, {dev.default_samplerate:.0f} Hz)")
except AudioDeviceError as exc:
    print(f"Note: Could not enumerate physical devices ({exc}).")


Discovered 11 Physical Audio Input Device(s):

  [Index 0] Microsoft Sound Mapper - Input (2 ch, 44100 Hz)
  [Index 1] Microphone (Realtek(R) Audio) (4 ch, 44100 Hz)
  [Index 4] Primary Sound Capture Driver (2 ch, 44100 Hz)
  [Index 5] Microphone (Realtek(R) Audio) (4 ch, 44100 Hz)
  [Index 9] Microphone (Realtek(R) Audio) (2 ch, 48000 Hz)
  [Index 12] PC Speaker (Realtek HD Audio 2nd output with SST) (2 ch, 48000 Hz)
  [Index 13] Microphone (Realtek HD Audio Mic input) (2 ch, 44100 Hz)
  [Index 14] Microphone 1 (Realtek HD Audio Mic input with SST) (2 ch, 48000 Hz)
  [Index 15] Microphone 2 (Realtek HD Audio Mic input with SST) (4 ch, 16000 Hz)
  [Index 18] PC Speaker (Realtek HD Audio output with SST) (2 ch, 48000 Hz)
  [Index 19] Stereo Mix (Realtek HD Audio Stereo input) (2 ch, 48000 Hz)


In [15]:
# 6.4.2 Interactive Microphone Speech Capture & Conversation
from typing import Optional
import ipywidgets as widgets

RECORD_SECONDS = 3.5
DEVICE_INDEX = None  # None uses system default microphone, or specify index from list above

mic_status = widgets.Output()
record_button = widgets.Button(
    description="Speak to Skeleton (3.5s)",
    button_style="danger",
    tooltip="Click to record from your microphone",
    icon="microphone",
)

async def _do_record_and_chat(duration: float = 3.5, device: Optional[int] = None):
    mic_status.clear_output()
    with mic_status:
        print(f"[RECORDING] Speak now ({duration:.1f} seconds)...")
    
    try:
        if "PYTEST_CURRENT_TEST" in os.environ:
            # Headless test runner deterministic fast path
            test_pipe = SkeletonAudioPipeline(
                stt_client=stt_client,
                tts_client=tts_client,
                orchestrator=orchestrator,
                mouth_sync=mouth_sync,
                recorder=MockAudioRecorder(),
            )
            resp = await test_pipe.record_and_process(duration_seconds=1.0, vision_context=vision_state)
        else:
            resp = await audio_pipeline.record_and_process(
                duration_seconds=duration,
                device_index=device,
                vision_context=vision_state,
            )
        
        with mic_status:
            print("Audio captured and transcribed!")
            print(f"You said: \"{resp.user_transcript}\"")
            print(f"Skeleton: \"{resp.skeleton_response}\"")
            print(f"Total Latency: {resp.total_latency_ms:.1f} ms | Jaw Frames: {len(resp.mouth_frames)}")
            display(Audio(resp.audio_bytes, autoplay=False))
        return resp
    except Exception as exc:
        with mic_status:
            print(f"[Error] Microphone capture failed: {exc}")
        return None

def _on_record_clicked(b):
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = asyncio.get_event_loop()
    loop.create_task(_do_record_and_chat(RECORD_SECONDS, DEVICE_INDEX))

record_button.on_click(_on_record_clicked)

# Display interactive button and status area
display(record_button, mic_status)

# Execute immediately on cell run for direct feedback
await _do_record_and_chat(RECORD_SECONDS, DEVICE_INDEX)


Button(button_style='danger', description='Speak to Skeleton (3.5s)', icon='microphone', style=ButtonStyle(), …

Output()

AudioDialogueResponse(user_transcript='Thank you.', skeleton_response="Don't mention it. That green sweatshirt and orange bucket combination makes you look like a walking pumpkin patch anyway.", audio_bytes=b'RIFF\xff\xff\xff\xffWAVEfmt \x10\x00\x00\x00\x01\x00\x01\x00\xc0]\x00\x00\x80\xbb\x00\x00\x02\x00\x10\x00data\xff\xff\xff\xff#\x00 \x00\x18\x004\x00I\x00;\x00=\x00C\x00C\x00;\x00M\x00g\x00L\x008\x003\x003\x00-\x00$\x00\'\x003\x002\x009\x009\x00A\x00e\x00b\x00@\x006\x00H\x00N\x00.\x00\x1c\x00\x1c\x00\n\x00\x04\x00\x19\x00)\x00B\x00E\x00;\x008\x00A\x00;\x00&\x00\x1f\x005\x000\x00\x1d\x00\x05\x00\xed\xff\x02\x00\x1e\x00\x16\x00\xfe\xff#\x00a\x00U\x000\x00\x01\x00\xf8\xff\x1f\x002\x00\x06\x00\xd9\xff\xf4\xff+\x00\x02\x00\xde\xff\xf0\xff\x0c\x00\x1a\x00\x1b\x00\x14\x00\x19\x00\x1e\x00.\x00\x01\x00\xd6\xff\xe1\xff\xea\xff\xf1\xff\r\x00\x02\x00\xea\xff\xf0\xff\x01\x00\xfa\xff\x04\x00\xff\xff\xfa\xff\x07\x00\xe2\xff\xe5\xff\xf7\xff\xee\xff\xe2\xff\xb5\xff\xa8\xff\xbf\xff\xbc\xff\xd3\xff\x